# Лекция 2. PyTorch и PyTorch Lightning — код

Код из лекции: тензоры и автоматическое дифференцирование, `Dataset` и `DataLoader`, цикл обучения, Lightning.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все».

## Введение

### Как запустить код

В Colab PyTorch и scikit-learn уже установлены — доставляем только Lightning. На своём компьютере вместо этой ячейки выполните в терминале `pip install torch lightning scikit-learn`.

In [ ]:
%pip install -q lightning

**Что получилось.** Если ячейка завершилась без ошибок, Lightning установлен.

## Глава 1. Тензоры и автоматическое дифференцирование

### 1.1. Что такое тензор в PyTorch

Чем тензор отличается от массива NumPy: он **запоминает, как был вычислен**. Посмотрим, что это значит.

In [ ]:
import torch

a = torch.tensor(2.0, requires_grad=True)   # параметр: за ним следим
b = torch.tensor(3.0)                       # обычные данные
c = a * b
print(c.grad_fn)                 # узел графа: c получен умножением
print(c.grad_fn.next_functions)  # ссылки на предыдущие узлы — по одной на каждый вход
print(b.grad_fn)                 # у обычных данных узла нет
c.backward()                     # обратный проход (подробно — в разделе 1.5)
print(a.grad)                    # производная c по a равна b — PyTorch сохранил его в узле

* `MulBackward0` — узел графа: `c` получен умножением.
* `next_functions` — ссылки назад, по одной на каждый вход умножения. `AccumulateGrad` — служебный узел листа `a` (туда попадёт производная); `None` — за `b` PyTorch не следит. Число `0` рядом — служебный номер.
* `None` — у обычных данных `b` узла нет.
* `tensor(3.)` — производная $\partial c/\partial a = b = 3$: PyTorch сохранил значение `b` в узле.

Числа вида `0x…` — адреса в памяти, у вас они будут другими.

### 1.2. Создание тензоров

In [ ]:
import numpy as np
import torch

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # из обычного списка Python
z = torch.zeros(2, 3)                         # таблица 2×3 из нулей
r = torch.randn(4, 2)                         # 4×2 случайных числа из нормального распределения N(0, 1)
s = torch.arange(0, 10, 2)                    # числа от 0 до 10 с шагом 2: 0, 2, 4, 6, 8
print(a)
print(a.shape, a.dtype, a.device)
print(s, s.dtype)

**Что получилось.** `torch.tensor` сам выбрал тип: дробные числа стали `float32`, целые — `int64`. Форма `torch.Size([2, 2])` — две строки, два столбца.

In [ ]:
x_np = np.array([1.0, 2.0, 3.0])
t_shared = torch.from_numpy(x_np)   # общая память с массивом NumPy
t_copy = torch.tensor(x_np)         # независимая копия
x_np[0] = 100.0                     # меняем исходный массив
print(t_shared)
print(t_copy)

**Что получилось.** `t_shared` изменился вслед за массивом (общая память), `t_copy` — нет (копия). Оба получили тип `float64`, как у NumPy, а слоям PyTorch нужен `float32` — поэтому тип обычно задают явно.

### 1.3. Операции и broadcasting

Повторим вычисление скрытого слоя сети 2–2–1 из лекции 1 — сразу для всех четырёх входов XOR.

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # 4 примера по 2 признака: форма (4, 2)
W = torch.tensor([[0.5, 0.2], [-0.3, 0.4]])                 # веса скрытого слоя: форма (2, 2)
b = torch.tensor([0.1, -0.2])                               # смещения двух нейронов: форма (2,)
Z = X @ W + b            # broadcasting: b прибавляется к каждой строке
A = torch.sigmoid(Z)
print(Z.shape)
print(A)

**Что получилось.** Третья строка — вход $(1, 0)$: $(0.6457, 0.5)$, ровно как на бумаге в лекции 1. Вектор `b` формы `(2,)` прибавился к каждой строке матрицы `(4, 2)` — это broadcasting.

**Broadcasting** — правило, по которому складываются тензоры разной формы.

In [ ]:
M = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])   # 3 примера × 2 нейрона: форма (3, 2)
v = torch.tensor([10., 20.])                        # смещения двух нейронов: форма (2,)
print(M + v)        # v прибавляется к каждой строке
print(M + 1)        # число прибавляется к каждому элементу
col = torch.tensor([[1.], [2.], [3.]])              # столбец: форма (3, 1)
print(col + v)      # «таблица сумм»: каждый элемент столбца с каждым элементом v

* `M + v` — вектор прибавился к каждой строке.
* `M + 1` — число прибавилось к каждому элементу.
* `col + v` — растянулись оба тензора: получилась таблица всех попарных сумм.

Не все формы совместимы. Вот что будет, если размеры не подходят:

In [ ]:
try:
    M + torch.tensor([1., 2., 3.])                  # (3, 2) + (3,) — несовместимо
except RuntimeError as e:
    print(e)

**Что получилось.** Размер 2 у первого тензора не совпадает с размером 3 у второго, и ни один не равен 1 — растянуть нельзя. Увидев такую ошибку, напечатайте `.shape` обоих тензоров.

**Изменение формы:** `reshape` перекладывает те же числа в таблицу другой формы.

In [ ]:
t = torch.arange(6)               # 6 чисел подряд, форма (6,)
print(t)
print(t.reshape(2, 3))            # те же числа: 2 строки по 3
print(t.reshape(3, 2))            # те же числа: 3 строки по 2
print(t.reshape(-1, 2).shape)     # -1 означает «посчитай сам»

**Что получилось.** Числа не меняются — они раскладываются по порядку, строка за строкой. Количество должно сохраниться: 6 = 2·3 = 3·2. `-1` означает «вычисли этот размер сам».

Зачем это нужно: картинку перед подачей в полносвязную сеть «разворачивают» в строку.

In [ ]:
images = torch.zeros(32, 8, 8)    # батч: 32 картинки по 8×8 пикселей
flat = images.reshape(32, -1)     # каждая картинка — строка из 8·8 = 64 чисел
print(images.shape, "→", flat.shape)

**Что получилось.** Номер картинки (32) сохранился, а два измерения 8×8 слились в одно: 64 числа на картинку.

### 1.4. Где живёт тензор: CPU и GPU

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.randn(3, 3).to(device)
print(t.device)

**Что получилось.** `cpu` — видеокарты нет. В Colab можно включить GPU: «Среда выполнения → Сменить среду выполнения → GPU», тогда здесь будет `cuda:0`.

### 1.5. Автоматическое дифференцирование

Главная возможность PyTorch. Один нейрон из раздела 3.5 лекции 1: вход $x = (1, 0)$, ответ $y = 1$, веса $w = (0.5, -0.3)$, смещение $b = 0.1$. Вручную мы получили $L \approx 0.126$ и $\partial L/\partial w_1 \approx -0.162$.

In [ ]:
x = torch.tensor([1.0, 0.0])
y = torch.tensor(1.0)
w = torch.tensor([0.5, -0.3], requires_grad=True)   # «следи за этим тензором»
b = torch.tensor(0.1, requires_grad=True)

z = w @ x + b                # прямой проход: взвешенная сумма
a = torch.sigmoid(z)         # активация
loss = (a - y) ** 2          # потеря

loss.backward()              # обратный проход — одна строка

print(f"loss = {loss.item():.3f}")
print("dL/dw =", w.grad)
print("dL/db =", b.grad)

**Что получилось.** Числа совпали с ручным расчётом из лекции 1: $-0.162$ для $w_1$ и ноль для $w_2$ (вход $x_2 = 0$ «молчит»). `-0.0` — это тот же ноль. Ни одной производной мы не выписывали — их посчитал `loss.backward()`.

Посмотрим на граф вычислений, который PyTorch записал во время прямого прохода.

In [ ]:
print(loss.grad_fn)
print(loss.grad_fn.next_functions)

* `PowBackward0` — узел потери: возведение в квадрат.
* `next_functions` — одна ссылка назад, к `SubBackward0`: квадрат применялся к результату вычитания `a - y`. Ссылка одна, потому что показатель `2` — обычное число, за ним PyTorch не следит.

Дальше цепочка продолжается: `Sub → Sigmoid → Add → Dot` — от потери назад к весам.

Проверим на «чаше» $f = w_1^2 + w_2^2$ из лекции 1: в точке $(3, 4)$ градиент должен быть $(6, 8)$.

In [ ]:
v = torch.tensor([3.0, 4.0], requires_grad=True)   # точка A = (w1, w2)
f = (v ** 2).sum()                                  # f = w1² + w2² = 25
f.backward()
print(f.item(), v.grad)

**Что получилось.** $f = 25$, градиент $(6, 8)$ — как на бумаге в лекции 1.

### 1.6. Три ловушки autograd

**Ловушка 1.** Градиенты накапливаются.

In [ ]:
w2 = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    loss2 = w2 ** 2          # производная: 2·w = 4
    loss2.backward()
    print(f"шаг {i}: w.grad = {w2.grad.item()}")

**Что получилось.** Правильная производная — 4, но после трёх вызовов в `.grad` лежит 12: `backward()` **прибавляет** к старому значению. Поэтому перед каждым новым `backward()` градиенты обнуляют.

**Ловушка 2.** Когда градиенты не нужны, граф можно не строить.

In [ ]:
with torch.no_grad():
    zz = w @ x + b
print(zz.requires_grad)      # False — граф не строился

**Что получилось.** `False` — внутри `torch.no_grad()` граф не строится. Так делают при проверке модели и при её использовании.

**Ловушка 3.** Шаг спуска не должен попасть в граф. Повторим спуск по «чаше» из лекции 1, $\eta = 0.05$.

In [ ]:
v = torch.tensor([3.0, 4.0], requires_grad=True)
eta = 0.05
for step in range(3):
    f = (v ** 2).sum()                  # прямой проход: f = w1² + w2²
    f.backward()                        # градиент → v.grad
    with torch.no_grad():               # шаг спуска не записываем в граф
        v -= eta * v.grad
    v.grad.zero_()                      # обнуляем перед следующим шагом
    print(f"шаг {step}: f = {f.item():.2f}, новые w = {v.detach().numpy().round(3)}")

**Что получилось.** Те же числа, что в таблице лекции 1: $25 \to 20.25 \to 16.40$, веса $(3, 4) \to (2.7, 3.6) \to (2.43, 3.24)$. Градиентный спуск — без единой выписанной производной.

## Глава 2. Dataset и DataLoader

### 2.2. Подготовка данных

Загружаем датасет Wine, делим на обучающую и валидационную выборки, стандартизируем признаки.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = load_wine()
X, y = wine.data, wine.target          # X: 178 вин × 13 признаков; y: 178 меток — сорт каждого вина (0, 1 или 2)
print(X.shape, y.shape)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # среднее и разброс считаем ТОЛЬКО по train
X_val = scaler.transform(X_val)           # и применяем их же к валидации
print(len(X_train), len(X_val))

**Что получилось.** 178 вин по 13 признаков. Разбиение 80/20: 142 вина для обучения, 36 для проверки. Стандартизация настроена только по обучающей выборке — так нет утечки данных.

### 2.3. Dataset: контракт из двух методов

Свой `Dataset`: два метода — сколько примеров и как достать $i$-й.

In [ ]:
class WineDataset(Dataset):
    def __init__(self, features, labels):
        # превращаем массивы NumPy в тензоры нужных типов
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_ds = WineDataset(X_train, y_train)
val_ds = WineDataset(X_val, y_val)

x0, y0 = train_ds[0]            # первое вино обучающей выборки
print(len(train_ds), x0.shape, x0.dtype, y0, y0.dtype)

**Что получилось.** 142 примера; один пример — 13 признаков типа `float32` и метка класса типа `int64`.

### 2.4. DataLoader: батчи, перемешивание, параллельность

`DataLoader` режет датасет на батчи и перемешивает их. Пройдём одну эпоху.

In [ ]:
torch.manual_seed(42)        # фиксируем случайность перемешивания
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(len(train_loader), len(val_loader))     # число батчей за эпоху
for xb, yb in train_loader:                   # одна эпоха: проходим все батчи
    print(xb.shape, yb.shape)

**Что получилось.** 5 батчей за эпоху: четыре по 32 вина и последний неполный из 14 (142 = 4·32 + 14). Каждый батч — матрица признаков и вектор меток. Валидационные 36 вин помещаются в один батч из 64.

## Глава 3. Цикл обучения

### 3.1. Модель: nn.Module

Модель — наследник `nn.Module`: 13 признаков → 32 нейрона с ReLU → 3 логита.

In [ ]:
from torch import nn

class WineMLP(nn.Module):
    def __init__(self, n_features=13, n_hidden=32, n_classes=3):
        super().__init__()                      # обязательная строка: «настрой родителя»
        self.net = nn.Sequential(
            nn.Linear(n_features, n_hidden),    # 13 → 32
            nn.ReLU(),
            nn.Linear(n_hidden, n_classes),     # 32 → 3; на выходе — логиты, без softmax
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = WineMLP()
print(model)
print("параметров:", sum(p.numel() for p in model.parameters()))
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

**Что получилось.** 547 параметров: $13 \cdot 32 + 32 = 448$ в первом слое и $32 \cdot 3 + 3 = 99$ во втором. Форма весов `(32, 13)` — `nn.Linear` хранит матрицу как «выходы × входы».

### 3.2. Функция потерь: CrossEntropyLoss

**Многоклассовая кросс-энтропия** — минус логарифм вероятности правильного класса. Посчитаем её по формуле для примера «кошка / собака / хомяк» из лекции 1.

In [ ]:
scores = torch.tensor([1.0, 0.0, -1.0])    # оценки модели: кошка, собака, хомяк
p = torch.softmax(scores, dim=0)           # вероятности (dim=0 — единственное измерение)
print(p)
print(-torch.log(p[0]))                    # если на картинке кошка (класс 0)
print(-torch.log(p[2]))                    # если на картинке хомяк (класс 2)

**Что получилось.** Вероятности $(0.67, 0.24, 0.09)$ — как в лекции 1. Если на картинке кошка, потеря $-\ln 0.67 \approx 0.41$ — модель права. Если хомяк, $-\ln 0.09 \approx 2.41$ — модель уверенно ошиблась, потеря большая.

Функция потерь для классификации. Посчитаем её на последнем батче из предыдущей ячейки.

In [ ]:
criterion = nn.CrossEntropyLoss()
logits = model(xb)                 # последний батч из 2.4: 14 вин × 3 оценки (по одной на сорт), форма (14, 3)
loss = criterion(logits, yb)       # yb — 14 номеров правильных сортов (0, 1 или 2): форма (14,), тип long
print(logits.shape, f"{loss.item():.4f}")

**Что получилось.** Около 1.08 — близко к $\ln 3 \approx 1.099$: необученная модель даёт трём сортам примерно равные вероятности. Хорошая проверка любой новой модели: начальная потеря ≈ $\ln K$, где $K$ — число классов.

### 3.3. Оптимизатор

Оптимизатор будет обновлять веса и обнулять градиенты.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

**Что получилось.** Ячейка ничего не печатает: мы создали оптимизатор Adam со скоростью обучения 0.001 для всех параметров модели.

### 3.4. Пять шагов цикла обучения

Пять шагов цикла обучения: `zero_grad` → прямой проход → потеря → `backward` → `step`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
model = WineMLP().to(device)               # модель — на выбранное устройство
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_one_epoch(model, loader):
    model.train()                                  # режим обучения (раздел 3.5)
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)      # батч — туда же, где модель
        optimizer.zero_grad()                      # 1. обнулить старые градиенты
        logits = model(xb)                         # 2. прямой проход
        loss = criterion(logits, yb)               # 3. потеря
        loss.backward()                            # 4. обратный проход (autograd)
        optimizer.step()                           # 5. шаг оптимизатора
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)        # средняя потеря за эпоху


@torch.no_grad()                                   # на проверке граф не нужен
def evaluate(model, loader):
    model.eval()                                   # режим проверки (раздел 3.5)
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * len(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


for epoch in range(1, 31):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    if epoch == 1 or epoch % 5 == 0:
        print(f"эпоха {epoch:2d} | train loss {train_loss:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_acc:.3f}")

* `train loss` и `val loss` — потери на обучающих и валидационных данных, `val acc` — доля правильно определённых сортов.
* После первой эпохи точность 58% — лучше угадывания (33%), но плохо. К 20-й эпохе модель верно определяет все 36 вин, которых не видела.
* Обе потери падают вместе — переобучения нет.

На GPU последние знаки могут немного отличаться.

### 3.6. Сохранение и загрузка модели

Сохраняем веса в файл и загружаем их в новую модель.

In [ ]:
torch.save(model.state_dict(), "wine_mlp.pt")       # сохранить веса в файл

model2 = WineMLP()                                   # пустая модель той же архитектуры
model2.load_state_dict(torch.load("wine_mlp.pt", map_location="cpu"))
model2.eval()
print(list(model.state_dict().keys()))

**Что получилось.** Сохраняются только числа — словарь «имя параметра → тензор». Поэтому при загрузке сначала создают модель того же класса.

### 3.7. XOR из лекции 1 — в десять строк

Сеть из лекции 1, решавшая XOR, — теперь на PyTorch.

In [ ]:
Xx = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
yx = torch.tensor([[0.], [1.], [1.], [0.]])

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1), nn.Sigmoid())
opt = torch.optim.SGD(net.parameters(), lr=2.0)
for epoch in range(5000):
    opt.zero_grad()
    loss = nn.functional.mse_loss(net(Xx), yx)    # MSE, как в лекции 1
    loss.backward()
    opt.step()
    if epoch % 1000 == 0:
        print(f"эпоха {epoch:5d} | loss = {loss.item():.4f}")
print(net(Xx).detach().round(decimals=3))

**Что получилось.** Ответы близки к 0, 1, 1, 0 — XOR решён. Те же пять шагов, та же MSE, что в лекции 1, но без ручного `backward`.

## Глава 4. LightningModule и DataModule

### 4.2. LightningModule

Та же модель, но в виде `LightningModule`: остаются только прямой проход, потеря и оптимизатор.

In [ ]:
import lightning as L

class WineClassifier(L.LightningModule):
    def __init__(self, n_features=13, n_hidden=32, n_classes=3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()            # запомнить аргументы __init__ в чекпоинте
        self.net = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_classes),
        )
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        xb, yb = batch
        loss = self.criterion(self(xb), yb)
        self.log("train_loss", loss, on_epoch=True, on_step=False)
        return loss                            # backward и step Lightning сделает сам

    def validation_step(self, batch, batch_idx):
        xb, yb = batch
        logits = self(xb)
        self.log("val_loss", self.criterion(logits, yb), prog_bar=True)
        self.log("val_acc", (logits.argmax(dim=1) == yb).float().mean(), prog_bar=True)

    def test_step(self, batch, batch_idx):
        xb, yb = batch
        self.log("test_acc", (self(xb).argmax(dim=1) == yb).float().mean())

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

**Что получилось.** Ячейка ничего не печатает: мы только описали класс модели. В `training_step` нет `zero_grad`, `backward` и `step` — их вызовет Lightning.

### 4.3. LightningDataModule

`LightningDataModule` собирает всю подготовку данных в один класс. Здесь три выборки: 70/15/15.

In [ ]:
class WineDataModule(L.LightningDataModule):
    def __init__(self, batch_size=32):
        super().__init__()
        self.batch_size = batch_size

    def prepare_data(self):
        # скачивание/распаковка — один раз; Wine встроен в sklearn, качать нечего
        pass

    def setup(self, stage=None):
        X, y = load_wine(return_X_y=True)
        X_train, X_rest, y_train, y_rest = train_test_split(       # 70% — обучение
            X, y, test_size=0.3, random_state=42, stratify=y)
        X_val, X_test, y_val, y_test = train_test_split(           # остаток пополам
            X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest)
        scaler = StandardScaler().fit(X_train)       # статистики — только по train
        self.train_ds = WineDataset(scaler.transform(X_train), y_train)
        self.val_ds = WineDataset(scaler.transform(X_val), y_val)
        self.test_ds = WineDataset(scaler.transform(X_test), y_test)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size)

**Что получилось.** Ячейка ничего не печатает: класс данных описан. Подготовка выполнится, когда `Trainer` вызовет `setup`.

### 4.4. Trainer: запуск обучения

`Trainer` выполняет цикл обучения за нас. Колбэки сохраняют лучшую модель и останавливают обучение, если валидационная потеря перестала падать.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

L.seed_everything(42)                  # фиксируем случайность в Python, NumPy и PyTorch
dm = WineDataModule(batch_size=32)
model = WineClassifier(lr=1e-3)

checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
early_stop = EarlyStopping(monitor="val_loss", mode="min", patience=10)

trainer = L.Trainer(
    max_epochs=100,
    accelerator="auto",          # GPU, если есть, иначе CPU
    devices=1,
    callbacks=[checkpoint, early_stop],
)
trainer.fit(model, datamodule=dm)
trainer.test(model, datamodule=dm, ckpt_path="best")

Что смотреть в выводе:

* `GPU available` — какое устройство выбрал `accelerator="auto"`.
* Таблица модели — те же 547 параметров.
* `max_epochs=100 reached` — ранняя остановка не сработала: валидационная потеря падала до конца.
* `test_acc` — точность на тестовых винах, которых модель не видела. У нас получилось 1.0, на валидации — около 0.963.

На вашей машине последние знаки могут немного отличаться.

Восстанавливаем лучшую модель из чекпоинта.

In [ ]:
best = WineClassifier.load_from_checkpoint(checkpoint.best_model_path)
print(best.hparams)

**Что получилось.** Гиперпараметры восстановились из чекпоинта сами — благодаря `save_hyperparameters()`.

### 4.5. Что Lightning делает за вас

**Что Lightning делает за вас** — упрощённая схема `trainer.fit`. Это псевдокод для понимания, запускать его не нужно (функции `move_to_device` не существует):

```python
# псевдокод: упрощённая схема trainer.fit
for epoch in range(max_epochs):
    model.train()
    for batch_idx, batch in enumerate(train_dataloader):
        batch = move_to_device(batch)
        optimizer.zero_grad()
        loss = model.training_step(batch, batch_idx)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(val_dataloader):
            model.validation_step(move_to_device(batch), batch_idx)

    for callback in callbacks:          # чекпоинт, ранняя остановка, ...
        callback.on_validation_end(trainer, model)
```